In [8]:
import train 
from utils import preprocess
from utils import models



In [ ]:
df_customers, df_products, df_transactions = preprocess.load_complete_dataset_filtered_number_customers(500, random_state=42)
df_transactions = preprocess.filter_customers_by_activity(df_transactions, min_purchases=3, max_months_since_last_purchase=12)

In [ ]:
df_train, df_test, eval_users, actual = train.leave_one_out_split(df_transactions)
print(f"eval_users: {len(eval_users)} | train: {len(df_train):,} | test: {len(df_test):,}")

In [ ]:
resultado_prep = models.xgboost_preprocess(
    df_customers, df_products, df_train,
    n_negativos_faciles=train.N_NEGATIVOS_FACILES,
    n_negativos_dificiles=train.N_NEGATIVOS_DIFICILES,
    random_state=train.RANDOM_STATE,
)
# Ajusta el desempaquetado según cuántos valores devuelva tu versión actual de xgboost_preprocess
X, y, sample_weight, dataset, article_df, user_df, categorical_categories = resultado_prep

feature_cols = list(X.columns)
categorical_cols_train = X.select_dtypes(include=['category']).columns.tolist()
print(f"X: {X.shape} | positivos: {(y==1).sum():,} | negativos: {(y==0).sum():,}")

In [6]:
import inspect
print(inspect.getsource(models.generar_negativos_cliente))

def generar_negativos_cliente(
    positivos_cliente, # DataFrame con los artículos que compró el cliente
    cliente,
    compras_por_cliente,
    todos_los_articulos,
    prob_muestreo,
    rng,
    mapa_articulo_cluster=None,
    articulos_por_cluster=None,
    n_negativos_faciles=4,
    n_negativos_dificiles=4
):
    # Usamos un set para las búsquedas rápidas (O(1))
    comprados = set(compras_por_cliente.get(cliente, set()))
    negativos = []

    # ==========================================
    # 1. NEGATIVOS FÁCILES (Globales por popularidad)
    # ==========================================
    n_faciles_necesarios = len(positivos_cliente) * n_negativos_faciles
    batch_size = min(n_faciles_necesarios * 3, len(todos_los_articulos))
    n_generados = 0
    intentos = 0
    
    while n_generados < n_faciles_necesarios and intentos < n_faciles_necesarios * 20:
        candidatos = rng.choice(todos_los_articulos, size=batch_size, p=prob_muestreo, replace=False)
        intentos +

In [7]:
import numpy as np
import time

rng = np.random.default_rng(42)
cluster_grande = np.arange(30000)  # simula un cluster grande

t0 = time.time()
for _ in range(1000):
    rng.choice(cluster_grande, size=20, replace=False)
print("replace=False:", time.time() - t0)

t0 = time.time()
for _ in range(1000):
    rng.choice(cluster_grande, size=20, replace=True)
print("replace=True: ", time.time() - t0)

replace=False: 0.009096622467041016
replace=True:  0.014106512069702148
